In [3]:
!pip install -q -U google-genai pydantic

import json
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata
from pydantic import BaseModel, Field

# Load API key from Google Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-3.5-flash-lite"

print("Gemini Client initialized successfully!")

Gemini Client initialized successfully!


In [4]:
zero_shot_prompt = """
Role: You are a Senior Data Analyst mentoring a junior analyst.
Context: A retail company noticed a 15% drop in monthly repeat orders.
Task: Provide 3 actionable SQL or BI investigation steps to find the root cause.
Constraints:
- Use bullet points only.
- Keep each bullet under 25 words.
- Do not include introductory or concluding text.
"""

response_zero = client.models.generate_content(
    model=MODEL_ID,
    contents=zero_shot_prompt
)
print("--- ZERO-SHOT WITH CONSTRAINTS ---")
print(response_zero.text)

few_shot_prompt = """
Classify the customer feedback into [Billing, Technical, Delivery] and assign Priority [High, Medium, Low].

Examples:
Feedback: "My dashboard won't load after the latest update and throws a 500 error."
Output: Category: Technical | Priority: High

Feedback: "Can I get a copy of last month's invoice?"
Output: Category: Billing | Priority: Low

Feedback: "My package arrived two days late, though everything inside is fine."
Output: Category: Delivery | Priority: Medium

Now classify this:
Feedback: "I was charged twice for my subscription today and cannot access my account."
Output:
"""

response_few = client.models.generate_content(
    model=MODEL_ID,
    contents=few_shot_prompt
)
print("\n--- FEW-SHOT CLASSIFICATION ---")
print(response_few.text)

--- ZERO-SHOT WITH CONSTRAINTS ---
* Query monthly repeat orders by product category to identify if the drop is isolated to specific inventory or universal.
* Build a retention cohort dashboard in your BI tool to check if the decline stems from newer or older customer cohorts.
* Analyze payment gateway error logs and checkout funnel drop-offs by device type to detect technical friction preventing completed reorders.

--- FEW-SHOT CLASSIFICATION ---
Category: Billing | Priority: High


In [6]:
import time

class TicketAnalysis(BaseModel):
    summary: str = Field(description="One-sentence summary of the text")
    category: str = Field(description="Category: Bug, Feature Request, or Account")
    sentiment: str = Field(description="Positive, Neutral, or Negative")
    action_required: bool = Field(description="True if immediate follow-up is needed")

raw_customer_note = """
Hi team, I've been using your analytics portal for three months and love the export speed.
However, since yesterday morning, whenever I filter by date range and export to CSV,
the currency column turns into #VALUE! errors. Please fix this before our Friday board review!
"""

for attempt in range(3):
    try:
        response_json = client.models.generate_content(
            model=MODEL_ID,
            contents=f"Analyze this customer message and extract structured details:\n{raw_customer_note}",
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=TicketAnalysis,
            ),
        )
        break
    except Exception:
        if attempt == 2:
            raise
        time.sleep(2)

structured_data = json.loads(response_json.text)
print("--- STRUCTURED JSON OUTPUT ---")
print(json.dumps(structured_data, indent=2))

--- STRUCTURED JSON OUTPUT ---
{
  "summary": "The customer experiences #VALUE! errors in currency columns when exporting filtered date ranges to CSV since yesterday morning.",
  "category": "Bug",
  "sentiment": "Negative",
  "action_required": true
}


In [7]:
buggy_code = """
def calculate_average_sales(sales_list):
    total = 0
    for item in sales_list
        total += item['amount']
    return total / len(sales_list)
"""

debug_prompt = f"""
Identify all syntax and runtime edge-case bugs in the Python function below,
then provide the corrected, production-safe code with docstrings:

{buggy_code}
"""

debug_response = client.models.generate_content(
    model=MODEL_ID,
    contents=debug_prompt
)
print("--- AI-ASSISTED DEBUGGING ---")
print(debug_response.text)

evaluation_results = [
    {
        "Prompt_ID": "P-01",
        "Technique": "Zero-Shot + Constraints",
        "Task": "Root Cause Analysis",
        "Format_Compliance": "Pass (3 bullets, <25 words)",
        "Accuracy_Score_1_to_5": 5,
        "Notes": "Strictly followed negative constraints with zero fluff."
    },
    {
        "Prompt_ID": "P-02",
        "Technique": "Few-Shot Prompting",
        "Task": "Ticket Classification",
        "Format_Compliance": "Pass (Exact delimiter match)",
        "Accuracy_Score_1_to_5": 5,
        "Notes": "Matched example schema accurately for multi-issue input."
    },
    {
        "Prompt_ID": "P-03",
        "Technique": "JSON Schema Enforcement",
        "Task": "Summarization & Extraction",
        "Format_Compliance": "Pass (Valid JSON)",
        "Accuracy_Score_1_to_5": 5,
        "Notes": "100% parseable via json.loads() without markdown wrappers."
    },
    {
        "Prompt_ID": "P-04",
        "Technique": "Role + Chain-of-Thought",
        "Task": "Python Debugging",
        "Format_Compliance": "Pass (Code + Explanation)",
        "Accuracy_Score_1_to_5": 5,
        "Notes": "Caught missing colon and ZeroDivisionError edge case."
    }
]

eval_df = pd.DataFrame(evaluation_results)
print("\n--- PROMPT TESTING & EVALUATION REPORT ---")
display(eval_df)

--- AI-ASSISTED DEBUGGING ---
### Identified Bugs and Edge Cases

1. **Syntax Error:** 
   - Missing a colon (`:`) at the end of the `for item in sales_list` statement.

2. **Runtime Edge Case: Division by Zero (`ZeroDivisionError`)**
   - If `sales_list` is empty, `len(sales_list)` is `0`, resulting in a crash when attempting `total / 0`.

3. **Runtime Edge Case: Type Error (`TypeError`)**
   - If `sales_list` is `None` instead of an iterable, calling `len()` or iterating over it will raise a `TypeError`.

4. **Runtime Edge Case: Key Error (`KeyError`)**
   - If any dictionary inside `sales_list` is missing the `'amount'` key, a `KeyError` will be raised.

5. **Runtime Edge Case: Type Error (`TypeError`)**
   - If an element in `sales_list` is not a dictionary (e.g., an `int`, `str`, or `None`), doing `item['amount']` will raise a `TypeError`.

6. **Runtime Edge Case: Type Error (`TypeError`)**
   - If the value of `'amount'` is not a number (e.g., a string like `"100"`), the addition

,Prompt_ID,Technique,Task,Format_Compliance,Accuracy_Score_1_to_5,Notes
0,P-01,Zero-Shot + Constraints,Root Cause Analysis,"Pass (3 bullets, <25 words)",5,Strictly followed negative constraints with ze...
1,P-02,Few-Shot Prompting,Ticket Classification,Pass (Exact delimiter match),5,Matched example schema accurately for multi-is...
2,P-03,JSON Schema Enforcement,Summarization & Extraction,Pass (Valid JSON),5,100% parseable via json.loads() without markdo...
3,P-04,Role + Chain-of-Thought,Python Debugging,Pass (Code + Explanation),5,Caught missing colon and ZeroDivisionError edg...


In [9]:
import time


class AIProductivityToolkit:
    """Reusable Prompt Engineering Toolkit for everyday professional workflows."""

    def __init__(self, client, model_id=MODEL_ID):
        self.client = client
        self.model_id = model_id

    def _run(self, prompt: str) -> str:
        for attempt in range(4):
            try:
                return self.client.models.generate_content(
                    model=self.model_id,
                    contents=prompt
                ).text
            except Exception:
                if attempt == 3:
                    raise
                time.sleep(2 * (attempt + 1))

    def summarize_document(self, text: str, bullet_count: int = 3) -> str:
        prompt = f"Summarize the text below into {bullet_count} crisp executive bullet points:\n\n{text}"
        return self._run(prompt)

    def draft_professional_email(self, recipient_role: str, topic: str, tone: str = "polite and concise") -> str:
        prompt = f"""
        Write a professional email to a {recipient_role} regarding: {topic}.
        Tone: {tone}.
        Include a clear Subject line, concise body (under 120 words), and a call to action.
        """
        return self._run(prompt)

    def tailor_resume_bullets(self, raw_experience: str, target_role: str) -> str:
        prompt = f"""
        Rewrite the following project experience into 3 ATS-friendly resume bullet points
        tailored for a {target_role} role using strong action verbs and measurable impact:
        {raw_experience}
        """
        return self._run(prompt)

    def research_and_code_assistant(self, topic: str) -> str:
        prompt = f"""
        Provide a concise technical breakdown of '{topic}' including:
        1. Core concept (2 sentences)
        2. Minimal working Python snippet
        3. One common pitfall to avoid
        """
        return self._run(prompt)


toolkit = AIProductivityToolkit(client)

print("1. EMAIL WORKFLOW:\n", toolkit.draft_professional_email(
    recipient_role="Internship Mentor",
    topic="Submitting Module 02 Prompt Engineering deliverables and requesting feedback"
))

print("\n2. RESUME BULLET WORKFLOW:\n", toolkit.tailor_resume_bullets(
    raw_experience="Built SQL queries and Power BI dashboards to track customer orders and help desk tickets.",
    target_role="Data Analyst"
))

print("\n3. SUMMARIZATION WORKFLOW:\n", toolkit.summarize_document(
    text="Prompt engineering improves LLM reliability by defining clear roles, supplying few-shot examples, enforcing JSON schemas, and validating outputs against test cases before production deployment.",
    bullet_count=2
))

print("\n4. RESEARCH & CODING WORKFLOW:\n", toolkit.research_and_code_assistant(
    topic="Python List Comprehensions with Conditional Filtering"
))

1. EMAIL WORKFLOW:
 Subject: Submission: Module 02 Prompt Engineering Deliverables - [Your Name]

Dear [Mentor Name],

I hope this email finds you well. 

I have completed and attached the deliverables for Module 02: Prompt Engineering. This assignment helped me further develop my understanding of structuring effective prompts and optimizing model outputs. 

When you have a moment, I would greatly appreciate your feedback on my work so I can continue to improve. 

Thank you for your time and ongoing guidance.

Best regards,

[Your Name]  
[Your Contact Information/Internship ID]

2. RESUME BULLET WORKFLOW:
 Here are 3 ATS-friendly resume bullet points tailored for a Data Analyst role, featuring strong action verbs, technical keywords, and measurable impact (using professional placeholders for metrics):

* **Engineered** complex SQL queries to extract and aggregate multi-source data, optimizing customer order tracking workflows and reducing manual reporting time by **[X]%**.
* **Designe